# NAFAS Gabès — Water Quality Contamination Classifier

Run all cells top-to-bottom. Works on **Kaggle**, **Google Colab**, and **local Jupyter**.

**Quickest path:** Upload this repo's `datasets/ml/` folder and the `UNEP GEMSWater` folder as a Kaggle dataset, then run here.

In [ ]:
# ── 0. Install dependencies ───────────────────────────────────────────────
!pip install lightgbm imbalanced-learn shap optuna fastapi uvicorn pyarrow -q

In [ ]:
# ── 1. Environment detection + path setup ────────────────────────────────
import os, sys, shutil
from pathlib import Path

def detect_env():
    if os.path.exists('/kaggle/input'):
        return 'kaggle'
    try:
        import google.colab  # noqa
        return 'colab'
    except ImportError:
        return 'local'

ENV = detect_env()
print(f'Environment: {ENV}')

# ── Set these to match where your data lives ──────────────────────────────
if ENV == 'kaggle':
    # Upload the GEMS folder as a Kaggle dataset named 'gems-water'
    # and the ml/ scripts as a dataset named 'nafas-ml-scripts'
    GEMS_DIR   = Path('/kaggle/input/gems-water')
    SCRIPTS_DIR = Path('/kaggle/input/nafas-ml-scripts')
    WORK_DIR   = Path('/kaggle/working')

elif ENV == 'colab':
    # Option A (recommended): share GEMS folder via Google Drive and get a
    # shareable folder link, then use gdown below.
    # Option B: leave GEMS_DIR pointing to your already-mounted path.
    GEMS_DIR   = Path('/content/gems_water')   # populated by gdown cell below
    SCRIPTS_DIR = Path('/content/nafas_ml')    # populated by upload cell below
    WORK_DIR   = Path('/content/nafas_ml')

else:  # local
    GEMS_DIR   = Path('../UNEP GEMSWater Global Freshwater Quality Archive')
    SCRIPTS_DIR = Path('.')
    WORK_DIR   = Path('.')

WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print(f'GEMS_DIR   : {GEMS_DIR}')
print(f'WORK_DIR   : {WORK_DIR}')

In [ ]:
# ── 2a. [COLAB ONLY] Download GEMS data via gdown ────────────────────────
# Share your GEMS folder on Google Drive → Get link → paste the folder ID below.
# Example link: https://drive.google.com/drive/folders/1AbCdEfGhIjKl  → ID = 1AbCdEfGhIjKl
#
# Skip this cell on Kaggle or local.

if ENV == 'colab':
    GDRIVE_FOLDER_ID = 'PASTE_YOUR_FOLDER_ID_HERE'  # <-- edit this

    if GDRIVE_FOLDER_ID == 'PASTE_YOUR_FOLDER_ID_HERE':
        print('[SKIP] Set GDRIVE_FOLDER_ID above, then re-run this cell.')
    else:
        !pip install gdown -q
        import gdown
        gdown.download_folder(id=GDRIVE_FOLDER_ID, output=str(GEMS_DIR), quiet=False, use_cookies=False)
        print('Download complete.')
else:
    print(f'[SKIP] Not Colab (env={ENV})')

In [ ]:
# ── 2b. [COLAB ONLY] Upload ml/ scripts via file picker ──────────────────
# Upload 01_preprocess.py, 02_train.py, 03_evaluate.py, 04_export_api.py
# (from the datasets/ml/ folder in the repo)
#
# Skip this cell on Kaggle or local.

if ENV == 'colab':
    from google.colab import files
    print('Select the 4 Python scripts from datasets/ml/ on your machine...')
    uploaded = files.upload()  # opens file picker
    for fname, data in uploaded.items():
        dest = WORK_DIR / fname
        dest.write_bytes(data)
        print(f'Saved {dest}')
else:
    print(f'[SKIP] Not Colab (env={ENV})')

In [ ]:
# ── 2c. [KAGGLE ONLY] Copy scripts from input dataset ────────────────────
if ENV == 'kaggle':
    for f in ['01_preprocess.py', '02_train.py', '03_evaluate.py', '04_export_api.py']:
        src = SCRIPTS_DIR / f
        if src.exists():
            shutil.copy(src, WORK_DIR / f)
            print(f'Copied {f}')
        else:
            print(f'[WARN] {src} not found — check your Kaggle dataset name')
else:
    print(f'[SKIP] Not Kaggle (env={ENV})')

In [ ]:
# ── 3. Preprocessing ─────────────────────────────────────────────────────
import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    return spec, mod

spec, preprocess = load_module('preprocess', WORK_DIR / '01_preprocess.py')
spec.loader.exec_module(preprocess)
preprocess.DATA_DIR = GEMS_DIR   # must come AFTER exec_module or it gets reset
preprocess.OUT_DIR  = WORK_DIR
preprocess.main()

print('\nParquet written to:', WORK_DIR / 'nafas_features.parquet')

In [ ]:
# ── 4. Train ─────────────────────────────────────────────────────────────
spec, train_mod = load_module('train', WORK_DIR / '02_train.py')
spec.loader.exec_module(train_mod)
train_mod.DATA_PATH = WORK_DIR / 'nafas_features.parquet'
train_mod.OUT_DIR   = WORK_DIR
train_mod.main()

In [ ]:
# ── 5. Evaluate ──────────────────────────────────────────────────────────
spec, eval_mod = load_module('evaluate', WORK_DIR / '03_evaluate.py')
spec.loader.exec_module(eval_mod)
eval_mod.OUT_DIR = WORK_DIR
eval_mod.main()

In [ ]:
# ── 6. Show plots inline ─────────────────────────────────────────────────
from IPython.display import Image, display
for img in ['confusion_matrix.png', 'roc_curves.png', 'shap_importance.png']:
    path = WORK_DIR / img
    if path.exists():
        print(f'--- {img} ---')
        display(Image(filename=str(path)))
    else:
        print(f'[MISSING] {img}')

In [ ]:
# ── 7. Quick inference test ───────────────────────────────────────────────
import joblib, numpy as np

model       = joblib.load(WORK_DIR / 'nafas_model.pkl')
scaler      = joblib.load(WORK_DIR / 'nafas_scaler.pkl')
le          = joblib.load(WORK_DIR / 'nafas_label_encoder.pkl')
feature_cols = joblib.load(WORK_DIR / 'nafas_feature_cols.pkl')

# Simulate a Gabes coastal station with elevated lead + phosphorus
sample = {
    'pb': 0.025,    # above WHO limit 0.01 mg/L
    'cd': 0.002,
    'ni': 0.03,
    'hg': 0.0005,
    'cr': 0.02,
    'as': 0.008,
    'p': 0.15,      # elevated phosphorus (phosphate industry)
    'n_ox': 0.5,
    'n_other': 0.3,
    'temp': 24.0,
    'ph': 7.8,
    'dgas': 6.5,
    'optical': 12.0,
    'month': 7,
    'season_enc': 2,
    'lat': 33.88,
    'lon': 10.10,
    'elevation': 1.0,
    'water_type_enc': 2,
}

row = np.array([sample.get(c, 0.0) for c in feature_cols], dtype=np.float32).reshape(1, -1)
row_scaled = scaler.transform(row)
proba = model.predict_proba(row_scaled)[0]
pred  = le.classes_[np.argmax(proba)]

print(f'Prediction : {pred}')
for cls, p in zip(le.classes_, proba):
    bar = '█' * int(p * 30)
    print(f'  {cls:<14} {bar} {p:.3f}')

In [ ]:
# ── 8. Download artefacts (Colab) / they're already in /kaggle/working ───
ARTEFACTS = [
    'nafas_model.pkl', 'nafas_scaler.pkl',
    'nafas_label_encoder.pkl', 'nafas_feature_cols.pkl',
    'nafas_features.parquet',
    'confusion_matrix.png', 'roc_curves.png',
    'shap_importance.png', 'feature_importance.csv',
]

if ENV == 'colab':
    from google.colab import files
    for f in ARTEFACTS:
        p = WORK_DIR / f
        if p.exists():
            files.download(str(p))
        else:
            print(f'[SKIP] {f} not found')
else:
    print('Artefacts are in:', WORK_DIR.resolve())
    for f in ARTEFACTS:
        p = WORK_DIR / f
        status = '✓' if p.exists() else '✗ MISSING'
        print(f'  {status}  {f}')